# MPX5700AP Pressure Sensor Calibration - Version 2

**Method**: Boyle's Law Independent Calibration
**Hardware**: M5Stack Atom Lite + MPX5700AP Pressure Sensor

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

print('Libraries imported!')

Libraries imported!


In [2]:
# Measured ADC values
adc_1ml = [875, 869, 883, 883, 834, 871, 863, 875, 868, 876]
adc_5ml = [1463, 1470, 1519, 1470, 1473, 1474, 1497, 1473, 1463, 1486]
adc_7ml = [2007, 1978, 1968, 1967, 1933, 1942, 1936, 1919, 1940, 1934]

compression_ml = np.array([1, 5, 7])
mean_adc = np.array([np.mean(adc_1ml), np.mean(adc_5ml), np.mean(adc_7ml)])
std_adc = np.array([np.std(adc_1ml, ddof=1), np.std(adc_5ml, ddof=1), np.std(adc_7ml, ddof=1)])

print('=== MEASURED ADC VALUES ===')
for i, comp in enumerate(compression_ml):
    print(f'{comp}ml: mean={mean_adc[i]:.1f}, std={std_adc[i]:.2f} ADC units')

=== MEASURED ADC VALUES ===
1ml: mean=869.7, std=14.04 ADC units
5ml: mean=1478.8, std=17.46 ADC units
7ml: mean=1952.4, std=26.78 ADC units


In [3]:
# Boyle's Law: P_gauge = P_atm * (V_initial / V_final) - P_atm
P_atm = 101.3  # kPa
V_initial = 10  # ml
V_final = V_initial - compression_ml
P_gauge_kPa = P_atm * (V_initial / V_final) - P_atm

print('=== BOYLE LAW PRESSURES ===')
for i, comp in enumerate(compression_ml):
    print(f'{comp}ml: V_final={V_final[i]:.0f}ml, P={P_gauge_kPa[i]:.2f} kPa')

=== BOYLE LAW PRESSURES ===
1ml: V_final=9ml, P=11.26 kPa
5ml: V_final=5ml, P=101.30 kPa
7ml: V_final=3ml, P=236.37 kPa


In [4]:
# Linear regression: ADC → kPa
slope_kPa_per_ADC, intercept_kPa, r_value, p_value, std_err = stats.linregress(mean_adc, P_gauge_kPa)

print('=== CALIBRATION ===')
print(f'Equation: P = {slope_kPa_per_ADC:.4f} × ADC + {intercept_kPa:.2f}')
print(f'R² = {r_value**2:.6f}')
print(f'\n1 ADC unit = {slope_kPa_per_ADC:.4f} kPa')

=== CALIBRATION ===
Equation: P = 0.2051 × ADC + -177.75
R² = 0.965403

1 ADC unit = 0.2051 kPa


In [5]:
# Noise analysis
std_kPa = std_adc * slope_kPa_per_ADC

print('=== NOISE LEVEL ===')
for i, comp in enumerate(compression_ml):
    print(f'{comp}ml: {std_adc[i]:.2f} ADC units = {std_kPa[i]:.3f} kPa')

=== NOISE LEVEL ===
1ml: 14.04 ADC units = 2.880 kPa
5ml: 17.46 ADC units = 3.581 kPa
7ml: 26.78 ADC units = 5.493 kPa


In [6]:
# Resolution (two-point method: 1ml and 7ml)
mean_adc_1ml, mean_adc_7ml = mean_adc[0], mean_adc[2]
std_adc_1ml, std_adc_7ml = std_adc[0], std_adc[2]

signal_diff_adc = abs(mean_adc_7ml - mean_adc_1ml)
combined_noise_adc = np.sqrt(std_adc_1ml**2 + std_adc_7ml**2)
num_steps = signal_diff_adc / combined_noise_adc
resolution_adc = combined_noise_adc
resolution_kPa = resolution_adc * slope_kPa_per_ADC
enob = np.log2(num_steps)

print('=== RESOLUTION (1ml & 7ml) ===')
print(f'Signal difference: {signal_diff_adc:.1f} ADC units')
print(f'Combined noise: {combined_noise_adc:.2f} ADC units')
print(f'Distinguishable steps: {num_steps:.1f}')
print(f'\nRESOLUTION: {resolution_adc:.2f} ADC units = {resolution_kPa:.3f} kPa')
print(f'ENOB: {enob:.2f} bits (theoretical: 12 bits)')

=== RESOLUTION (1ml & 7ml) ===
Signal difference: 1082.7 ADC units
Combined noise: 30.24 ADC units
Distinguishable steps: 35.8

RESOLUTION: 30.24 ADC units = 6.202 kPa
ENOB: 5.16 bits (theoretical: 12 bits)


In [7]:
print('='*70)
print('  ANSWERS TO SUPERVISOR QUESTIONS')
print('='*70)
print(f'\n1. ONE ADC UNIT = {slope_kPa_per_ADC:.4f} kPa')
print(f'   (Calculated using Boyle Law + linear regression)')
print(f'\n2. NOISE LEVEL:')
for i, comp in enumerate(compression_ml):
    print(f'   {comp}ml: {std_adc[i]:.2f} ADC units = {std_kPa[i]:.3f} kPa')
print(f'\n3. MINIMUM DETECTABLE CHANGE:')
print(f'   ADC must change by ≥{resolution_adc:.0f} units')
print(f'   This equals {resolution_kPa:.2f} kPa')
print(f'\n   If ADC changes less than {resolution_adc:.0f} units,')
print(f'   it could be noise, not real pressure change.')
print('\n' + '='*70)

  ANSWERS TO SUPERVISOR QUESTIONS

1. ONE ADC UNIT = 0.2051 kPa
   (Calculated using Boyle Law + linear regression)

2. NOISE LEVEL:
   1ml: 14.04 ADC units = 2.880 kPa
   5ml: 17.46 ADC units = 3.581 kPa
   7ml: 26.78 ADC units = 5.493 kPa

3. MINIMUM DETECTABLE CHANGE:
   ADC must change by ≥30 units
   This equals 6.20 kPa

   If ADC changes less than 30 units,
   it could be noise, not real pressure change.

